# SyntenyScan &nbsp;v1.1

Finds conserved gene order (**synteny**) between any two genomes at NCBI,
and the recent duplications (**paralogy**) inside each one.

Nothing is installed on your computer. Google’s machine does the work.

## How to run it

1. Edit **Step 1** if you want a different pair of species. Otherwise leave it.
2. From the menu choose **Runtime → Run all**.
3. Wait. The slow part is Step 5, usually 10–30 minutes.
4. Step 10 hands you a ZIP file of all the results.

If the page says *Sign in*, sign in with any Google account. It is free.

## What you get back

| File | Contents |
|---|---|
| `blocks.tsv` | every collinear block, with its chromosomes and size |
| `anchors.tsv` | every matched gene pair inside those blocks |
| `tandem.tsv` | genes sitting in tandem arrays |
| `summary.txt` | the headline counts |
| `dotplot.png` / `.pdf` | the whole-genome comparison figure |

---

## Step 1. Choose the two genomes

Two dropdown lists appear beside this cell. Pick a crop in each, **or**
leave them on `ask me` and the notebook will show you a numbered menu and
wait for your answer.

Every accession on the list carries a RefSeq annotation, which is what this
pipeline reads. Five Indian pulses — urdbean, lentil, field bean, horse
gram and moth bean — have no such genome at NCBI yet, and the notebook
names them rather than failing halfway through a download.

In [ ]:
# @title Step 1 &nbsp; Choose the two genomes &nbsp; { display-mode: "both" }

# @markdown Pick a crop in each list below, **or** leave them on `ask me` and
# @markdown the notebook will show you a numbered menu when you run it.

SPECIES_A = "ask me"  # @param ["ask me", "pigeonpea", "soybean", "mungbean", "adzukibean", "cowpea", "chickpea", "commonbean", "medicago", "groundnut", "pea", "I will type an accession"]
SPECIES_B = "ask me"  # @param ["ask me", "pigeonpea", "soybean", "mungbean", "adzukibean", "cowpea", "chickpea", "commonbean", "medicago", "groundnut", "pea", "I will type an accession"]

# @markdown ---
# @markdown **Advanced.** Leave these alone to reproduce the published run.
EVALUE = "1e-10"  # @param ["1e-10", "1e-5", "1e-20"]
MAX_TARGETS = 25  # @param {type:"integer"}
HITS_PER_SPECIES = 5  # @param {type:"integer"}


# ---- crops with a RefSeq-annotated genome at NCBI ------------------------
# Each accession was checked to carry RefSeq protein annotation, which is what
# this pipeline reads. A GCA_ (GenBank-only) record will not work.
#          name            accession           code  species
CROPS = {
    "pigeonpea":  ("GCF_000340665.2", "cc", "Cajanus cajan"),
    "soybean":    ("GCF_000004515.6", "gm", "Glycine max"),
    "mungbean":   ("GCF_000741045.1", "vr", "Vigna radiata"),
    "adzukibean": ("GCF_001190045.1", "va", "Vigna angularis"),
    "cowpea":     ("GCF_004118075.2", "vu", "Vigna unguiculata"),
    "chickpea":   ("GCF_000331145.1", "ca", "Cicer arietinum"),
    "commonbean": ("GCF_000499845.2", "pv", "Phaseolus vulgaris"),
    "medicago":   ("GCF_003473485.1", "mt", "Medicago truncatula"),
    "groundnut":  ("GCF_003086295.3", "ah", "Arachis hypogaea"),
    "pea":        ("GCF_024323335.1", "ps", "Pisum sativum"),
}

# ---- crops with NO RefSeq-annotated genome at NCBI -----------------------
# Named so the notebook can say so plainly instead of failing mid-download.
NO_REFSEQ = {
    "urdbean": "Vigna mungo", "blackgram": "Vigna mungo",
    "lentil": "Lens culinaris",
    "lablab": "Lablab purpureus", "fieldbean": "Lablab purpureus",
    "horsegram": "Macrotyloma uniflorum",
    "mothbean": "Vigna aconitifolia",
}

import re

# Printed on every run so you can always see WHICH version of the
# notebook you are looking at - Colab loads it from GitHub, so an old
# copy in the repository is served until the file there is replaced.
VERSION = "1.1"
print(f"SyntenyScan v{VERSION}")
print()

_NAMES = list(CROPS)
_ACC = re.compile(r"GCF_\d+\.\d+")


def _menu():
    print("=" * 66)
    print("  Which two genomes do you want to compare?")
    print("=" * 66)
    for i, k in enumerate(_NAMES, 1):
        acc, _, latin = CROPS[k]
        print(f"  {i:>3}  {k:<11} {latin:<22} {acc}")
    print(f"  {len(_NAMES) + 1:>3}  something else — I will type an NCBI accession")
    print()
    print("  Not available: urdbean, lentil, field bean, horse gram and moth")
    print("  bean have no RefSeq-annotated genome at NCBI yet.")
    print()


def _ask(label, default_i, taken=None):
    """Ask for one genome. Returns (accession, code, species name)."""
    n = len(_NAMES)
    while True:
        try:
            raw = input(f"{label} genome  "
                        f"[Enter = {_NAMES[default_i - 1]}]: ").strip()
        except (EOFError, OSError):
            print(f"  (no keyboard here — using {_NAMES[default_i - 1]})")
            return CROPS[_NAMES[default_i - 1]]

        if raw == "":
            choice = default_i
        elif raw.isdigit():
            choice = int(raw)
        else:                       # a name or an accession typed directly
            key = raw.lower().replace(" ", "").replace("_", "").replace("-", "")
            if key in CROPS:
                return CROPS[key]
            if key in NO_REFSEQ:
                print(f"  {raw} ({NO_REFSEQ[key]}) has no RefSeq-annotated genome "
                      f"at NCBI yet. Pick another.")
                continue
            if _ACC.fullmatch(raw):
                return (raw, "s1" if label == "FIRST" else "s2", raw)
            print(f"  '{raw}' is not on the list and is not a GCF_ accession. "
                  f"Type a number from 1 to {n + 1}.")
            continue

        if choice == n + 1:
            try:
                acc = input("  NCBI RefSeq accession (e.g. GCF_000340665.2): ").strip()
            except (EOFError, OSError):
                continue
            if _ACC.fullmatch(acc):
                return (acc, "s1" if label == "FIRST" else "s2", acc)
            print("  That is not a GCF_ accession. Find one at "
                  "https://www.ncbi.nlm.nih.gov/datasets/genome/")
            continue
        if not 1 <= choice <= n:
            print(f"  Type a number from 1 to {n + 1}.")
            continue
        got = CROPS[_NAMES[choice - 1]]
        if taken and got[0] == taken:
            print("  That is the same genome as the first one. Pick a different crop.")
            continue
        return got


def _from_dropdown(choice, slot):
    """Resolve a value chosen in the form above. None means 'ask me'."""
    key = str(choice).strip().lower().replace(" ", "").replace("_", "").replace("-", "")
    if key in ("askme", "iwilltypeanaccession"):
        return None
    if key in CROPS:
        return CROPS[key]
    if key in NO_REFSEQ:
        raise SystemExit(
            f"\n{choice} ({NO_REFSEQ[key]}) has no RefSeq-annotated genome at NCBI "
            f"yet.\nThis pipeline reads RefSeq annotation, so it cannot use that "
            f"species.\nPick another crop, or check NCBI again later:\n"
            f"  https://www.ncbi.nlm.nih.gov/datasets/genome/?taxon="
            + NO_REFSEQ[key].replace(" ", "%20"))
    if _ACC.fullmatch(str(choice).strip()):
        return (choice.strip(), "s1" if slot == "A" else "s2", choice.strip())
    raise SystemExit(f"\n'{choice}' is not a crop on the list and is not a GCF_ "
                     f"accession.\nNames you can use: " + ", ".join(sorted(CROPS)))


_a = _from_dropdown(SPECIES_A, "A")
_b = _from_dropdown(SPECIES_B, "B")

if _a is None or _b is None:
    _menu()
if _a is None:
    _a = _ask("FIRST ", 1)
if _b is None:
    _b = _ask("SECOND", 2, taken=_a[0])

SPECIES_A_ACCESSION, SPECIES_A_CODE, SPECIES_A_NAME = _a
SPECIES_B_ACCESSION, SPECIES_B_CODE, SPECIES_B_NAME = _b

if SPECIES_A_ACCESSION == SPECIES_B_ACCESSION:
    raise SystemExit("\nBoth genomes are the same. Choose two different ones.")
if SPECIES_A_CODE == SPECIES_B_CODE:
    SPECIES_B_CODE = "s2"

print()
print("-" * 66)
print("  You chose")
print("-" * 66)
print(f"  A   {SPECIES_A_NAME:<24} {SPECIES_A_ACCESSION:<18} labelled {SPECIES_A_CODE}")
print(f"  B   {SPECIES_B_NAME:<24} {SPECIES_B_ACCESSION:<18} labelled {SPECIES_B_CODE}")
print()
print(f"  DIAMOND E-value {EVALUE} · max-target-seqs {MAX_TARGETS} · "
      f"{HITS_PER_SPECIES} hits per query per species")
print("-" * 66)
print("  Everything below this cell now runs on its own. The long step is the")
print("  protein comparison, usually 10 to 30 minutes.")

## Step 2. Install the software

DIAMOND, MCScanX and the NCBI download tool. Two to three minutes.

MCScanX is old C++ and will not compile on a modern compiler without a
small patch, so the cell adds the two missing `#include` lines before
building. That is a known quirk of the program, not a problem with your
setup.

In [ ]:
%%bash
set -e
mkdir -p /content/bin && cd /content

# ---- DIAMOND (precompiled) ------------------------------------------
if [ ! -x bin/diamond ]; then
  wget -q https://github.com/bbuchfink/diamond/releases/download/v2.1.9/diamond-linux64.tar.gz
  tar xzf diamond-linux64.tar.gz && mv diamond bin/ && rm diamond-linux64.tar.gz
fi

# ---- MCScanX (from source, with the include patch) -------------------
if [ ! -x bin/MCScanX ]; then
  rm -rf MCScanX && git clone -q https://github.com/wyp1125/MCScanX.git
  cd MCScanX
  # MCScanX predates modern C++ headers; add the two includes it assumes.
  # Guarded by grep so the cell is safe to run more than once.
  for f in msa.h dissect_multiple_alignment.h out_utils.h struct.h; do
    [ -f "$f" ] || continue
    grep -q "#include <string>"  "$f" || sed -i "1i #include <string>"  "$f"
    grep -q "#include <cstring>" "$f" || sed -i "1i #include <cstring>" "$f"
  done
  # quiet on success, loud on failure: the build emits many harmless
  # sprintf warnings that would otherwise look like errors
  make -s > /tmp/mcscanx_build.log 2>&1 || true
  if [ ! -x MCScanX ]; then
    echo "MCScanX BUILD FAILED. Last lines of the log:"; tail -20 /tmp/mcscanx_build.log; exit 1
  fi
  cp MCScanX duplicate_gene_classifier ../bin/ 2>/dev/null || cp MCScanX ../bin/
  cd ..
fi

# ---- NCBI datasets CLI ----------------------------------------------
if [ ! -x bin/datasets ]; then
  wget -q -O bin/datasets https://ftp.ncbi.nlm.nih.gov/pub/datasets/command-line/v2/linux-amd64/datasets
  chmod +x bin/datasets
fi

echo
./bin/diamond version
ls -l bin/MCScanX bin/datasets | awk '{print $NF, "OK"}'

## Step 3. Download the two genomes from NCBI

Protein sequences and the annotation file for each. A few minutes,
depending on genome size.

In [ ]:
import os, subprocess, glob, shutil
os.makedirs("/content/data", exist_ok=True)

def fetch(accession, code):
    out = f"/content/data/{code}"
    if os.path.exists(f"{out}/genomic.gff") and os.path.exists(f"{out}/protein.faa"):
        print(f"{code}: already downloaded"); return
    zipf = f"/content/data/{code}.zip"
    subprocess.run(["/content/bin/datasets", "download", "genome", "accession",
                    accession, "--include", "protein,gff3",
                    "--filename", zipf], check=True)
    subprocess.run(["unzip", "-qo", zipf, "-d", f"/content/data/{code}_raw"], check=True)
    os.makedirs(out, exist_ok=True)
    for name in ("genomic.gff", "protein.faa"):
        hits = glob.glob(f"/content/data/{code}_raw/ncbi_dataset/data/**/{name}",
                         recursive=True)
        assert hits, f"{name} not found for {accession} - does it have RefSeq annotation?"
        shutil.copy(hits[0], f"{out}/{name}")
    os.remove(zipf)
    print(f"{code}: downloaded {accession}")

fetch(SPECIES_A_ACCESSION, SPECIES_A_CODE)
fetch(SPECIES_B_ACCESSION, SPECIES_B_CODE)

for c in (SPECIES_A_CODE, SPECIES_B_CODE):
    for n in ("genomic.gff", "protein.faa"):
        mb = os.path.getsize(f"/content/data/{c}/{n}") / 1e6
        print(f"  {c}/{n:14} {mb:8.1f} MB")

## Step 4. One protein per gene

A gene can have many transcripts and so many protein sequences. Comparing
all of them would count the same gene several times, so this step keeps
exactly one protein per gene: the one with the longest coding sequence.

It also writes the position file MCScanX needs — which chromosome each
gene sits on, and where.

In [ ]:
import gzip, re, os, sys
from collections import defaultdict

def _attrs(s):
    d = {}
    for kv in s.rstrip(";").split(";"):
        if "=" in kv:
            k, v = kv.split("=", 1)
            d[k] = v
    return d

def _openf(p):
    return gzip.open(p, "rt") if p.endswith(".gz") else open(p)

def parse_gff(gff):
    """protein-coding genes, their longest-CDS protein, and seqid->chromosome."""
    genes, rna2gene, rna_prot = {}, {}, {}
    rna_cdslen = defaultdict(int)
    seqid_chrom = {}
    with _openf(gff) as fh:
        for line in fh:
            if line.startswith("#"):
                continue
            f = line.rstrip("\n").split("\t")
            if len(f) < 9:
                continue
            seqid, _, ftype, start, end, _, strand, _, attr = f
            if ftype == "region":
                a = _attrs(attr)
                if "chromosome" in a:
                    seqid_chrom[seqid] = a["chromosome"]
            elif ftype == "gene":
                a = _attrs(attr)
                if a.get("gene_biotype") != "protein_coding":
                    continue
                m = re.search(r"GeneID:(\d+)", a.get("Dbxref", ""))
                key = "LOC" + m.group(1) if m else a.get("ID", "")
                genes[key] = (seqid, int(start), int(end), strand)
            elif ftype == "mRNA":
                a = _attrs(attr)
                rid = a.get("ID")
                m = re.search(r"GeneID:(\d+)", a.get("Dbxref", ""))
                if rid and m:
                    rna2gene[rid] = "LOC" + m.group(1)
            elif ftype == "CDS":
                a = _attrs(attr)
                par = a.get("Parent")
                if par:
                    rna_cdslen[par] += int(end) - int(start) + 1
                    if "protein_id" in a:
                        rna_prot[par] = a["protein_id"]
    best = {}
    for rid, clen in rna_cdslen.items():
        g, p = rna2gene.get(rid), rna_prot.get(rid)
        if g and p and (g not in best or clen > best[g][1]):
            best[g] = (p, clen)
    return genes, best, seqid_chrom

def chrom_label(seqid, seqid_chrom, code):
    """cc01..cc11 for numbered chromosomes; ccSxxxx for everything else."""
    c = seqid_chrom.get(seqid)
    if c and re.fullmatch(r"\d+", c):
        return f"{code}{int(c):02d}"
    return f"{code}S" + seqid.split(".")[0].replace("_", "")

def prepare(code):
    d = f"/content/data/{code}"
    genes, best, seqid_chrom = parse_gff(f"{d}/genomic.gff")
    prot2gene = {v[0]: g for g, v in best.items()}
    os.makedirs("/content/work", exist_ok=True)
    with open(f"/content/work/{code}.gff", "w") as out:
        for g, (p, _) in best.items():
            if g in genes:
                seqid, s, e, _ = genes[g]
                out.write(f"{chrom_label(seqid, seqid_chrom, code)}\t{g}\t{s}\t{e}\n")
    n = 0
    with _openf(f"{d}/protein.faa") as fh, open(f"/content/work/{code}.faa", "w") as out:
        write = False
        for line in fh:
            if line.startswith(">"):
                acc = line[1:].split()[0]
                write = acc in prot2gene
                if write:
                    out.write(">" + prot2gene[acc] + "\n"); n += 1
            elif write:
                out.write(line)
    print(f"{code}: {len(genes):,} protein-coding genes, "
          f"{len(best):,} with a representative protein, {n:,} written")
    return len(genes), n

nA = prepare(SPECIES_A_CODE)
nB = prepare(SPECIES_B_CODE)

!cat /content/work/{SPECIES_A_CODE}.faa /content/work/{SPECIES_B_CODE}.faa > /content/work/both.faa
!cat /content/work/{SPECIES_A_CODE}.gff /content/work/{SPECIES_B_CODE}.gff > /content/work/both.gff
print("\ncombined:", sum(1 for l in open("/content/work/both.gff")), "genes")

## Step 5. Compare every protein with every other — DIAMOND

**This is the slow step: usually 10–30 minutes.** Leave the tab open.

Both species go into one database, so the search finds matches between the
two species and within each one at the same time. Matches within a species
are what reveal duplications.

In [ ]:
%%time
import subprocess, os
os.makedirs("/content/work", exist_ok=True)
THREADS = os.cpu_count() or 2

subprocess.run(["/content/bin/diamond", "makedb",
                "--in", "/content/work/both.faa",
                "-d", "/content/work/both", "--quiet"], check=True)

subprocess.run(["/content/bin/diamond", "blastp",
                "-q", "/content/work/both.faa",
                "-d", "/content/work/both.dmnd",
                "-o", "/content/work/both.blast",
                "-f", "6", "-e", EVALUE,
                "--max-target-seqs", str(MAX_TARGETS),
                "--threads", str(THREADS)], check=True)

n = sum(1 for _ in open("/content/work/both.blast"))
print(f"\nDIAMOND wrote {n:,} hit lines using {THREADS} threads")

## Step 6. Keep the best five hits per species

MCScanX expects roughly the top five matches per gene. Taking the top five
*overall* from a two-species database is a trap: a gene with several close
copies of itself lets those copies fill all five slots, so its matches in
the other species are thrown away and it can never be placed in a shared
block.

Since those same copies are what make a marker amplify more than one place,
that shortcut would manufacture part of the very pattern you are testing.
Splitting the quota **per target species** removes the problem.

In [ ]:
from collections import defaultdict

species = {}
for c in (SPECIES_A_CODE, SPECIES_B_CODE):
    for line in open(f"/content/work/{c}.gff"):
        species[line.split("\t")[1]] = c

kept = defaultdict(int)
n_in = n_out = 0
with open("/content/work/both.blast") as fh, \
     open("/content/work/both.filtered.blast", "w") as out:
    for line in fh:
        n_in += 1
        f = line.split("\t")
        key = (f[0], species.get(f[1]))
        if kept[key] < HITS_PER_SPECIES:
            kept[key] += 1
            out.write(line); n_out += 1

print(f"read  {n_in:,} hits")
print(f"kept  {n_out:,} hits  ({HITS_PER_SPECIES} per query per target species)")

## Step 7. Find the collinear blocks — MCScanX

MCScanX takes the matches plus each gene’s position and asks whether the
matching genes appear in the *same order* on both sides. A run of five or
more is a collinear block. Fast, a minute or two.

In [ ]:
import os, shutil, subprocess
os.makedirs("/content/mcscan", exist_ok=True)
shutil.copy("/content/work/both.gff", "/content/mcscan/both.gff")
shutil.copy("/content/work/both.filtered.blast", "/content/mcscan/both.blast")

r = subprocess.run(["/content/bin/MCScanX", "/content/mcscan/both"],
                   capture_output=True, text=True)
print(r.stdout[-1200:] or r.stderr[-1200:])
assert os.path.exists("/content/mcscan/both.collinearity"), "MCScanX produced no output"
print("\ncollinearity file written")

## Step 8. Turn the output into tables

MCScanX writes its own text format. This turns it into two spreadsheets:
one row per block, and one row per matched gene pair.

Each block is labelled by which species it joins — the two different
species (shared ancestry) or one species with itself (duplication).

In [ ]:
import re, os
import pandas as pd

pos = {}
for line in open("/content/work/both.gff"):
    c, g, s, e = line.rstrip("\n").split("\t")
    pos[g] = (c, int(s), int(e))

hdr = re.compile(r"^## Alignment (\d+): score=([\d.]+) e_value=(\S+) "
                 r"N=(\d+) (\S+)&(\S+) (plus|minus)")
blocks, anchors, bid = [], [], None
for line in open("/content/mcscan/both.collinearity"):
    if line.startswith("## Alignment"):
        m = hdr.match(line)
        bid = int(m.group(1))
        blocks.append(dict(block=bid, score=float(m.group(2)),
                           evalue=m.group(3), n_anchors=int(m.group(4)),
                           chr_a=m.group(5), chr_b=m.group(6),
                           orient=m.group(7)))
    elif not line.startswith("#"):
        f = line.rstrip("\n").split("\t")
        anchors.append(dict(block=bid, gene_a=f[1].strip(),
                            gene_b=f[2].strip(), evalue=f[3].strip()))

B, A = pd.DataFrame(blocks), pd.DataFrame(anchors)
B["sp_a"] = B.chr_a.str[:2]
B["sp_b"] = B.chr_b.str[:2]
B["kind"] = B.apply(lambda r: "-".join(sorted([r.sp_a, r.sp_b])), axis=1)

os.makedirs("/content/results", exist_ok=True)
B.to_csv("/content/results/blocks.tsv", sep="\t", index=False)
A.to_csv("/content/results/anchors.tsv", sep="\t", index=False)

# tandem arrays
tand = []
tf = "/content/mcscan/both.tandem"
if os.path.exists(tf):
    for line in open(tf):
        gs = [g for g in line.strip().split(",") if g]
        for g in gs:
            tand.append(dict(gene=g, species=g in pos and pos[g][0][:2] or "",
                             array=",".join(gs)))
T = pd.DataFrame(tand)
T.to_csv("/content/results/tandem.tsv", sep="\t", index=False)

# ---- summary ---------------------------------------------------------
AB = "-".join(sorted([SPECIES_A_CODE, SPECIES_B_CODE]))
AA, BB = f"{SPECIES_A_CODE}-{SPECIES_A_CODE}", f"{SPECIES_B_CODE}-{SPECIES_B_CODE}"
ab = A.merge(B[["block", "kind"]], on="block")
lines = [f"SyntenyScan summary",
         f"  A: {SPECIES_A_NAME} ({SPECIES_A_ACCESSION}) = {SPECIES_A_CODE}",
         f"  B: {SPECIES_B_NAME} ({SPECIES_B_ACCESSION}) = {SPECIES_B_CODE}", ""]
for kind, label in [(AB, "between the two species"),
                    (AA, f"{SPECIES_A_NAME} against itself"),
                    (BB, f"{SPECIES_B_NAME} against itself")]:
    nb = int((B.kind == kind).sum())
    na = int((ab.kind == kind).sum())
    big = int(B[B.kind == kind].n_anchors.max()) if nb else 0
    lines.append(f"{label:36} {nb:6,} blocks  {na:8,} anchors  largest {big:5,} genes")

genes_a = {g for g, v in pos.items() if v[0].startswith(SPECIES_A_CODE)}
syn = (set(ab[ab.kind == AB].gene_a) | set(ab[ab.kind == AB].gene_b)) & genes_a
lines += ["", f"{SPECIES_A_NAME} genes inside a shared block: "
              f"{len(syn):,} of {len(genes_a):,} ({100*len(syn)/len(genes_a):.1f}%)"]

open("/content/results/summary.txt", "w").write("\n".join(lines) + "\n")
print("\n".join(lines))

## Step 9. The dot plot

Every point is one matched gene pair, placed by where it sits on each
genome. Diagonal streaks are conserved blocks. If one genome has doubled
since the two split, each region of the other lines up against **two**
streaks — that pattern is visible at a glance.

In [ ]:
import re
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 8,
                     "figure.facecolor": "#fcfcfb", "savefig.facecolor": "#fcfcfb"})

def chroms(code):
    cs = sorted({v[0] for v in pos.values() if re.fullmatch(code + r"\d\d", v[0])})
    return cs

ca, cb = chroms(SPECIES_A_CODE), chroms(SPECIES_B_CODE)
if not ca or not cb:
    print("One genome has no numbered chromosomes (scaffold-level assembly),")
    print("so a chromosome dot plot cannot be drawn. The tables are still complete.")
else:
    lens = {c: max(v[2] for v in pos.values() if v[0] == c) for c in ca + cb}
    off, o = {}, 0
    for c in ca:
        off[c] = o; o += lens[c]
    tot_a = o
    o = 0
    for c in cb:
        off[c] = o; o += lens[c]
    tot_b = o

    d = ab[ab.kind == AB].copy()
    def side(g):
        return pos[g][0][:2] if g in pos else None
    d["ga"] = d.apply(lambda r: r.gene_a if side(r.gene_a) == SPECIES_A_CODE else r.gene_b, axis=1)
    d["gb"] = d.apply(lambda r: r.gene_b if side(r.gene_a) == SPECIES_A_CODE else r.gene_a, axis=1)
    d = d[d.ga.map(lambda g: pos.get(g, ("",))[0] in off) &
          d.gb.map(lambda g: pos.get(g, ("",))[0] in off)]
    x = d.ga.map(lambda g: off[pos[g][0]] + pos[g][1])
    y = d.gb.map(lambda g: off[pos[g][0]] + pos[g][1])

    fig, ax = plt.subplots(figsize=(7.2, 6.4))
    ax.scatter(x / 1e6, y / 1e6, s=0.6, c="#2166ac", alpha=0.45,
               linewidths=0, rasterized=True)
    for c in ca[1:]:
        ax.axvline(off[c] / 1e6, color="#d9d9d9", lw=0.5, zorder=0)
    for c in cb[1:]:
        ax.axhline(off[c] / 1e6, color="#d9d9d9", lw=0.5, zorder=0)
    ax.set_xticks([(off[c] + lens[c] / 2) / 1e6 for c in ca])
    ax.set_xticklabels([c[2:] for c in ca], fontsize=6.5)
    ax.set_yticks([(off[c] + lens[c] / 2) / 1e6 for c in cb])
    ax.set_yticklabels([c[2:] for c in cb], fontsize=6.5)
    ax.set_xlim(0, tot_a / 1e6); ax.set_ylim(0, tot_b / 1e6)
    ax.set_xlabel(f"{SPECIES_A_NAME} chromosomes", fontsize=8.5)
    ax.set_ylabel(f"{SPECIES_B_NAME} chromosomes", fontsize=8.5)
    ax.set_title(f"Collinear anchor pairs  ({len(d):,} anchors, "
                 f"{d.block.nunique():,} blocks)", fontsize=9, loc="left", pad=8)
    fig.tight_layout()
    for ext in ("png", "pdf"):
        fig.savefig(f"/content/results/dotplot.{ext}", dpi=300)
    print(f"dot plot: {len(d):,} anchors from {d.block.nunique():,} "
          f"chromosome-anchored blocks")
    from IPython.display import Image, display
    display(Image("/content/results/dotplot.png"))

## Step 10. Download everything

Packs the results into one ZIP and sends it to your computer. If the
download does not start, open the folder icon in the left sidebar and
right-click `results.zip`.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("/content/results", "zip", "/content/results")
!ls -lh /content/results.zip
files.download("/content/results.zip")

---

## If something goes wrong

**"MCScanX produced no output"** — almost always means the two genomes are
too distantly related to share detectable blocks, or one of them is a
scaffold-level assembly with no chromosomes. Check `summary.txt`.

**Session disconnected during Step 5** — Colab stops a notebook left idle.
Keep the tab open and visible. Re-running from the top is safe: every step
skips work it has already done.

**"not found for GCF_..."** — that genome has no RefSeq protein annotation.
Not every assembly at NCBI is annotated. Pick one whose NCBI page lists an
Annotation Release.

**Out of memory** — two very large genomes can exceed the free tier. Runtime
→ Change runtime type → High-RAM if your account offers it.

## What to cite

If you publish results from this notebook, cite the two programs it runs:

- Buchfink B, Xie C, Huson DH (2015) Fast and sensitive protein alignment
  using DIAMOND. Nat Methods 12:59–60. https://doi.org/10.1038/nmeth.3176
- Wang Y, Tang H, DeBarry JD, Tan X, Li J, Wang X, Lee TH, Jin H, Marler B,
  Guo H et al (2012) MCScanX: a toolkit for detection and evolutionary
  analysis of gene synteny and collinearity. Nucleic Acids Res 40:e49.
  https://doi.org/10.1093/nar/gkr1293

and the genome papers for whichever species you used.